# 04 - Exploratory Data Analysis: Financial Performance

Period comparisons use Smart automobile expansion (2010–2021), Transition Year (2022), and AI expansion (2023–2026). Years 2025–2026 remain flagged for source validation within AI expansion. Other historical trend and company-ranking analyses retain their explicitly stated 2010–2024 or 2022–2024 windows. All shares refer to this dataset; period labels do not establish causation.

In [1]:
from pathlib import Path
import pandas as pd

## Load, validate, and create analysis subsets

In [2]:
cwd = Path.cwd().resolve()
project_root = cwd if (cwd / "data" / "processed").is_dir() else cwd.parent
input_path = project_root / "data" / "processed" / "financials_features.csv"
if not input_path.is_file():
    raise FileNotFoundError(f"Feature dataset not found: {input_path}")
df = pd.read_csv(input_path)
df_snapshot = df.copy(deep=True)
expected_groups = {"Foundry", "Fabless", "IDM", "Equipment", "EDA Software"}
input_validation = {
    "row_count_617": len(df) == 617,
    "column_count_38": df.shape[1] == 38,
    "year_range_2010_2026": (int(df.year.min()), int(df.year.max())) == (2010, 2026),
    "unique_year_company": not df.duplicated(["year", "company_name"]).any(),
    "no_infinite_values": not df.select_dtypes(include="number").isin([float("inf"), float("-inf")]).any().any(),
    "all_five_value_chain_groups": set(df.value_chain_group.dropna()) == expected_groups,
}
for check, passed in input_validation.items():
    assert passed, f"Input validation failed: {check}"
df_historical = df.loc[df.year.le(2024)].copy()
df_smart_auto = df.loc[df.analysis_period.eq("Smart automobile expansion")].copy()
df_ai_expansion = df.loc[df.analysis_period.eq("AI expansion")].copy()
df_transition = df.loc[df.analysis_period.eq("Transition Year")].copy()
df_caution = df.loc[df.year.between(2025, 2026)].copy()
print(f"Input path: {input_path}")
display(pd.Series(input_validation, name="passed").to_frame())
display(df.groupby("analysis_period", sort=False).agg(rows=("year", "size"), minimum_year=("year", "min"), maximum_year=("year", "max"), source_review_rows=("source_validation_caution", "sum")))

Input path: C:\Users\ALBERT\Desktop\Year 1 sem2\DADS5001_Aj.Thitirat_SUN\Mid-Term Project\semiconductor company datasets\data\processed\financials_features.csv


,passed
row_count_617,True
column_count_38,True
year_range_2010_2026,True
unique_year_company,True
no_infinite_values,True
all_five_value_chain_groups,True


,rows,minimum_year,maximum_year,source_review_rows
analysis_period,,,,
Smart automobile expansion,417,2010,2021,0
Transition Year,40,2022,2022,0
AI expansion,160,2023,2026,80


## Dataset and entity coverage

Changes in total revenue may partly reflect changes in dataset coverage. Later entry into the dataset is descriptive coverage information, not a missing-data error.

In [3]:
coverage_by_year = df.groupby("year").agg(observation_count=("company_name", "size"), company_count=("company_name", "nunique"), segment_count=("segment", "nunique"), value_chain_group_count=("value_chain_group", "nunique"), total_revenue_usd_bn=("revenue_usd_bn", "sum"), source_validation_caution_rows=("source_validation_caution", "sum")).reset_index()
display(coverage_by_year)
company_coverage = df.groupby("company_name", as_index=False).agg(first_observed_year=("year", "min"), last_observed_year=("year", "max"), observed_year_count=("year", "nunique"), value_chain_group=("value_chain_group", "first"))
historical_years = set(range(2010, 2025))
pre_through_core_years = set(range(2018, 2025))
years_by_company = df_historical.groupby("company_name")["year"].agg(lambda values: set(values))
companies_every_2010_2024 = sorted(years_by_company[years_by_company.map(lambda years: historical_years.issubset(years))].index.tolist())
companies_every_2018_2024 = sorted(years_by_company[years_by_company.map(lambda years: pre_through_core_years.issubset(years))].index.tolist())
companies_entering_after_2018 = company_coverage.loc[company_coverage.first_observed_year.gt(2018), "company_name"].sort_values().tolist()
display(company_coverage)
print(f"Companies present every year 2010–2024 ({len(companies_every_2010_2024)}): {companies_every_2010_2024}")
print(f"Companies present throughout 2018–2024 ({len(companies_every_2018_2024)}): {companies_every_2018_2024}")
print(f"Companies entering after 2018 ({len(companies_entering_after_2018)}): {companies_entering_after_2018}")

,year,observation_count,company_count,segment_count,value_chain_group_count,total_revenue_usd_bn,source_validation_caution_rows
0,2010,33,33,19,5,235.00,0
1,2011,33,33,19,5,249.80,0
2,2012,33,33,19,5,266.67,0
3,2013,33,33,19,5,284.22,0
4,2014,33,33,19,5,292.77,0
5,2015,33,33,19,5,311.52,0
6,2016,33,33,19,5,369.27,0
7,2017,34,34,19,5,430.35,0
8,2018,35,35,19,5,476.20,0
9,2019,37,37,20,5,495.05,0


,company_name,first_observed_year,last_observed_year,observed_year_count,value_chain_group
0,AMD,2010,2026,17,Fabless
1,ASML,2010,2026,17,Equipment
2,Analog Devices,2010,2026,17,IDM
3,Applied Materials,2010,2026,17,Equipment
4,Broadcom,2010,2026,17,Fabless
5,Cadence,2010,2026,17,EDA Software
6,Cerebras Systems,2019,2026,8,Fabless
7,ChangXin Memory (CXMT),2018,2026,9,IDM
8,GlobalFoundries,2010,2026,17,Foundry
9,Graphcore,2019,2026,8,Fabless


Companies present every year 2010–2024 (33): ['AMD', 'ASML', 'Analog Devices', 'Applied Materials', 'Broadcom', 'Cadence', 'GlobalFoundries', 'HiSilicon (Huawei)', 'Hua Hong', 'Infineon', 'Intel', 'KLA', 'Lam Research', 'Marvell', 'MediaTek', 'Microchip', 'Micron', 'NVIDIA', 'NXP Semiconductors', 'Onsemi', 'Qualcomm', 'Renesas', 'SCREEN Holdings', 'SK Hynix', 'SMIC', 'STMicroelectronics', 'Samsung Foundry', 'Samsung Memory', 'Synopsys', 'TSMC', 'Texas Instruments', 'Tokyo Electron', 'UMC']
Companies present throughout 2018–2024 (35): ['AMD', 'ASML', 'Analog Devices', 'Applied Materials', 'Broadcom', 'Cadence', 'ChangXin Memory (CXMT)', 'GlobalFoundries', 'HiSilicon (Huawei)', 'Hua Hong', 'Infineon', 'Intel', 'KLA', 'Lam Research', 'Marvell', 'MediaTek', 'Microchip', 'Micron', 'NVIDIA', 'NXP Semiconductors', 'Onsemi', 'Qualcomm', 'Renesas', 'SCREEN Holdings', 'SK Hynix', 'SMIC', 'STMicroelectronics', 'Samsung Foundry', 'Samsung Memory', 'Synopsys', 'TSMC', 'Texas Instruments', 'Tokyo El

## Industry revenue trend, 2010–2024

In [4]:
industry_trend = df_historical.groupby("year").agg(total_revenue_within_dataset_usd_bn=("revenue_usd_bn", "sum"), company_count=("company_name", "nunique")).reset_index()
industry_trend["year_over_year_total_revenue_change_pct"] = industry_trend["total_revenue_within_dataset_usd_bn"].pct_change(fill_method=None).mul(100)
valid_industry_growth = industry_trend.dropna(subset=["year_over_year_total_revenue_change_pct"])
highest_growth_year = valid_industry_growth.loc[valid_industry_growth.year_over_year_total_revenue_change_pct.idxmax()]
lowest_growth_year = valid_industry_growth.loc[valid_industry_growth.year_over_year_total_revenue_change_pct.idxmin()]
display(industry_trend)
print(f"Highest annual growth: {int(highest_growth_year.year)} ({highest_growth_year.year_over_year_total_revenue_change_pct:.2f}%)")
print(f"Lowest annual growth: {int(lowest_growth_year.year)} ({lowest_growth_year.year_over_year_total_revenue_change_pct:.2f}%)")

,year,total_revenue_within_dataset_usd_bn,company_count,year_over_year_total_revenue_change_pct
0,2010,235.00,33,NaN
1,2011,249.80,33,6.297872
2,2012,266.67,33,6.753403
3,2013,284.22,33,6.581168
4,2014,292.77,33,3.008233
5,2015,311.52,33,6.404345
6,2016,369.27,33,18.538136
7,2017,430.35,34,16.540743
8,2018,476.20,35,10.654119
9,2019,495.05,37,3.958421


Highest annual growth: 2024 (26.62%)
Lowest annual growth: 2023 (-6.18%)


## Value-chain analysis

In [5]:
value_chain_yearly = df_historical.groupby(["year", "value_chain_group"]).agg(entity_count=("company_name", "nunique"), total_revenue_usd_bn=("revenue_usd_bn", "sum"), mean_operating_margin_pct=("operating_margin_pct", "mean"), median_operating_margin_pct=("operating_margin_pct", "median"), mean_rd_intensity_pct=("rd_intensity_pct", "mean"), median_rd_intensity_pct=("rd_intensity_pct", "median"), mean_capex_intensity_pct=("capex_intensity_pct", "mean"), median_capex_intensity_pct=("capex_intensity_pct", "median")).reset_index()
value_chain_yearly["value_chain_revenue_share_pct"] = value_chain_yearly["total_revenue_usd_bn"].div(value_chain_yearly.groupby("year")["total_revenue_usd_bn"].transform("sum")).mul(100)
display(value_chain_yearly.loc[value_chain_yearly.year.isin([2018, 2021, 2022, 2024])])

,year,value_chain_group,entity_count,total_revenue_usd_bn,mean_operating_margin_pct,median_operating_margin_pct,mean_rd_intensity_pct,median_rd_intensity_pct,mean_capex_intensity_pct,median_capex_intensity_pct,value_chain_revenue_share_pct
40,2018,EDA Software,2,5.22,40.100000,40.10,35.076253,35.076253,0.953159,0.953159,1.096178
41,2018,Equipment,6,57.62,30.833333,30.35,12.701197,12.050182,9.320836,9.983226,12.099958
42,2018,Fabless,7,77.11,31.257143,29.70,14.973603,11.983471,8.871326,9.990301,16.192776
43,2018,Foundry,6,61.88,39.450000,38.90,8.052530,8.021642,44.978411,44.990193,12.994540
44,2018,IDM,14,274.37,28.292857,30.30,11.868520,11.977383,21.361230,14.991452,57.616548
55,2021,EDA Software,2,7.67,40.700000,40.70,35.074674,35.074674,0.915277,0.915277,1.271910
56,2021,Equipment,6,81.85,30.550000,31.45,12.639008,11.989790,9.323973,9.995599,13.573122
57,2021,Fabless,12,127.36,24.366667,23.70,31.618428,19.988895,5.637949,9.971910,21.120011
58,2021,Foundry,6,101.05,40.716667,40.40,7.985970,7.998889,44.972641,44.970354,16.757044
59,2021,IDM,14,285.10,32.021429,37.05,11.862488,11.970906,21.341335,15.015664,47.277913


## Smart automobile expansion, Transition Year, and AI expansion

In [6]:
def summarize_period(frame, period_name):
    annual_group = frame.groupby(["year", "value_chain_group"])["revenue_usd_bn"].sum().rename("annual_group_revenue").reset_index()
    annual_total = frame.groupby("year")["revenue_usd_bn"].sum().rename("annual_total_revenue")
    annual_group = annual_group.join(annual_total, on="year")
    annual_group["annual_share_pct"] = annual_group.annual_group_revenue.div(annual_group.annual_total_revenue).mul(100)
    base = frame.groupby("value_chain_group").agg(median_entity_revenue_usd_bn=("revenue_usd_bn", "median"), mean_operating_margin_pct=("operating_margin_pct", "mean"), median_operating_margin_pct=("operating_margin_pct", "median"), mean_rd_intensity_pct=("rd_intensity_pct", "mean"), median_rd_intensity_pct=("rd_intensity_pct", "median"), mean_capex_intensity_pct=("capex_intensity_pct", "mean"), median_capex_intensity_pct=("capex_intensity_pct", "median"))
    base["average_annual_total_revenue_usd_bn"] = annual_group.groupby("value_chain_group").annual_group_revenue.mean()
    base["average_revenue_share_within_dataset_pct"] = annual_group.groupby("value_chain_group").annual_share_pct.mean()
    return base.add_prefix(period_name + "__")

pre_summary = summarize_period(df_smart_auto, "smart_auto")
core_summary = summarize_period(df_ai_expansion, "ai_expansion")
transition_summary = summarize_period(df_transition, "transition")
period_comparison = pre_summary.join(transition_summary).join(core_summary)
metric_names = [column.removeprefix("smart_auto__") for column in pre_summary.columns]
for metric in metric_names:
    pre_values = period_comparison[f"smart_auto__{metric}"]
    core_values = period_comparison[f"ai_expansion__{metric}"]
    period_comparison[f"absolute_difference__{metric}"] = core_values - pre_values
    safe_pre = pre_values.where(pre_values.ne(0))
    period_comparison[f"percentage_difference__{metric}"] = core_values.sub(pre_values).div(safe_pre).mul(100)
display(period_comparison)

,smart_auto__median_entity_revenue_usd_bn,smart_auto__mean_operating_margin_pct,smart_auto__median_operating_margin_pct,smart_auto__mean_rd_intensity_pct,smart_auto__median_rd_intensity_pct,smart_auto__mean_capex_intensity_pct,smart_auto__median_capex_intensity_pct,smart_auto__average_annual_total_revenue_usd_bn,smart_auto__average_revenue_share_within_dataset_pct,transition__median_entity_revenue_usd_bn,...,absolute_difference__median_rd_intensity_pct,percentage_difference__median_rd_intensity_pct,absolute_difference__mean_capex_intensity_pct,percentage_difference__mean_capex_intensity_pct,absolute_difference__median_capex_intensity_pct,percentage_difference__median_capex_intensity_pct,absolute_difference__average_annual_total_revenue_usd_bn,percentage_difference__average_annual_total_revenue_usd_bn,absolute_difference__average_revenue_share_within_dataset_pct,percentage_difference__average_revenue_share_within_dataset_pct
value_chain_group,,,,,,,,,,,,,,,,,,,,,
EDA Software,2.005,40.958333,41.00,35.007751,35.035492,1.003627,0.997661,4.415000,1.158595,4.170,...,-0.029005,-0.082787,-0.005909,-0.588813,0.013985,1.401791,6.842500,154.983012,0.104117,8.986526
Equipment,6.550,30.812500,31.20,12.668432,12.020842,9.332791,9.989075,44.427500,11.611783,16.395,...,-0.005746,-0.047802,0.010197,0.109258,0.007868,0.078761,70.185000,157.976479,1.307864,11.263246
Fabless,5.415,29.058333,28.40,17.080638,12.017973,8.457815,9.995088,65.396667,16.852252,3.925,...,10.485092,87.245094,-3.104283,-36.703128,-4.032155,-40.341371,256.645833,392.444824,17.600210,104.438326
Foundry,4.640,39.973611,39.95,7.989156,8.001869,44.984192,45.001390,52.639167,13.676192,8.700,...,-0.002054,-0.025663,0.018103,0.040243,-0.001390,-0.003088,86.873333,165.035541,1.778469,13.004123
IDM,8.760,27.269281,23.30,12.033956,11.982882,20.053680,14.973730,209.585833,56.701180,14.130,...,0.001681,0.014027,1.302415,6.494643,0.028080,0.187526,113.396667,54.105120,-20.790659,-36.667067


## Company growth and rankings, 2022–2024

Two-year CAGR is an EDA summary metric only and is not written back to the feature dataset. Percentage-growth rankings retain revenue-size context.

In [7]:
revenue_wide = df.loc[df.year.isin([2022, 2024])].pivot(index="company_name", columns="year", values="revenue_usd_bn").rename(columns={2022: "revenue_2022", 2024: "revenue_2024"})
eligible = revenue_wide.dropna().loc[lambda table: table.revenue_2022.gt(0) & table.revenue_2024.gt(0)].copy()
company_2024 = df.loc[df.year.eq(2024), ["company_name", "value_chain_group", "operating_margin_pct", "rd_intensity_pct", "capex_intensity_pct", "revenue_share_within_dataset_pct", "operating_income_review_flag"]].set_index("company_name")
company_2022 = df.loc[df.year.eq(2022), ["company_name", "rd_intensity_pct", "capex_intensity_pct"]].rename(columns={"rd_intensity_pct": "rd_intensity_2022_pct", "capex_intensity_pct": "capex_intensity_2022_pct"}).set_index("company_name")
company_growth = eligible.join(company_2024).join(company_2022)
company_growth["absolute_revenue_change_usd_bn"] = company_growth.revenue_2024 - company_growth.revenue_2022
company_growth["revenue_growth_2022_2024_pct"] = company_growth.revenue_2024.div(company_growth.revenue_2022).sub(1).mul(100)
company_growth["two_year_cagr_pct"] = company_growth.revenue_2024.div(company_growth.revenue_2022).pow(1 / 2).sub(1).mul(100)
company_growth = company_growth.reset_index()[["company_name", "value_chain_group", "revenue_2022", "revenue_2024", "absolute_revenue_change_usd_bn", "revenue_growth_2022_2024_pct", "two_year_cagr_pct", "operating_margin_pct", "rd_intensity_pct", "capex_intensity_pct", "revenue_share_within_dataset_pct", "operating_income_review_flag", "rd_intensity_2022_pct", "capex_intensity_2022_pct"]]
display(company_growth)
ranking_columns = ["company_name", "value_chain_group", "revenue_2022", "revenue_2024", "absolute_revenue_change_usd_bn", "revenue_growth_2022_2024_pct", "two_year_cagr_pct", "operating_margin_pct", "rd_intensity_pct", "capex_intensity_pct", "revenue_share_within_dataset_pct", "operating_income_review_flag"]
rankings = {
    "Top 10 by 2024 revenue": company_growth.nlargest(10, "revenue_2024"),
    "Top 10 by absolute revenue increase": company_growth.nlargest(10, "absolute_revenue_change_usd_bn"),
    "Top 10 by two-year CAGR": company_growth.nlargest(10, "two_year_cagr_pct"),
    "Bottom 10 by absolute revenue change": company_growth.nsmallest(10, "absolute_revenue_change_usd_bn"),
    "Top 10 by 2024 operating margin": company_growth.nlargest(10, "operating_margin_pct"),
    "Top 10 by 2024 R&D intensity": company_growth.nlargest(10, "rd_intensity_pct"),
    "Top 10 by 2024 CapEx intensity": company_growth.nlargest(10, "capex_intensity_pct"),
}
for title, table in rankings.items():
    print(title)
    display(table[ranking_columns])

,company_name,value_chain_group,revenue_2022,revenue_2024,absolute_revenue_change_usd_bn,revenue_growth_2022_2024_pct,two_year_cagr_pct,operating_margin_pct,rd_intensity_pct,capex_intensity_pct,revenue_share_within_dataset_pct,operating_income_review_flag,rd_intensity_2022_pct,capex_intensity_2022_pct
0,AMD,Fabless,23.00,26.05,3.05,13.260870,6.424090,23.6,20.000000,9.980806,3.144199,False,20.000000,10.000000
1,ASML,Equipment,23.90,31.81,7.91,33.096234,15.367341,33.3,16.001257,6.004401,3.839423,False,15.983264,5.983264
2,Analog Devices,IDM,12.20,9.47,-2.73,-22.377049,-11.896112,42.3,12.038015,12.038015,1.143016,False,11.967213,11.967213
3,Applied Materials,Equipment,27.32,27.31,-0.01,-0.036603,-0.018303,31.1,12.010253,9.996338,3.296279,False,12.005857,9.992679
4,Broadcom,Fabless,33.50,54.34,20.84,62.208955,27.361280,25.4,11.998528,9.992639,6.558762,False,12.000000,10.000000
5,Cadence,EDA Software,3.52,4.47,0.95,26.988636,12.689235,41.3,34.899329,0.894855,0.539523,False,34.943182,1.136364
6,Cerebras Systems,Fabless,0.22,0.77,0.55,250.000000,87.082869,4.3,50.649351,1.298701,0.092938,False,50.000000,0.000000
7,ChangXin Memory (CXMT),IDM,3.10,5.07,1.97,63.548387,27.886038,30.0,10.059172,34.911243,0.611942,False,10.000000,35.161290
8,GlobalFoundries,Foundry,8.00,6.54,-1.46,-18.250000,-9.584293,37.0,7.951070,44.954128,0.789369,False,8.000000,45.000000
9,Graphcore,Fabless,0.05,0.05,0.00,0.000000,0.000000,11.1,40.000000,0.000000,0.006035,False,40.000000,0.000000


Top 10 by 2024 revenue


,company_name,value_chain_group,revenue_2022,revenue_2024,absolute_revenue_change_usd_bn,revenue_growth_2022_2024_pct,two_year_cagr_pct,operating_margin_pct,rd_intensity_pct,capex_intensity_pct,revenue_share_within_dataset_pct,operating_income_review_flag
21,NVIDIA,Fabless,26.12,130.48,104.36,399.540582,123.504045,59.5,25.000000,2.000307,15.748754,False
34,TSMC,Foundry,74.93,88.89,13.96,18.630722,8.917731,41.3,7.998650,44.999438,10.728899,False
32,Samsung Memory,IDM,75.70,74.79,-0.91,-1.202114,-0.602874,26.8,10.001337,35.004680,9.027049,False
14,Intel,IDM,64.11,54.42,-9.69,-15.114647,-7.866752,8.8,19.992650,25.009188,6.568418,False
4,Broadcom,Fabless,33.50,54.34,20.84,62.208955,27.361280,25.4,11.998528,9.992639,6.558762,False
27,SK Hynix,IDM,34.69,51.53,16.84,48.544249,21.878730,28.8,9.994178,35.008733,6.219599,False
24,Qualcomm,Fabless,44.51,39.18,-5.33,-11.974837,-6.178274,31.1,11.995916,10.005105,4.728971,False
1,ASML,Equipment,23.90,31.81,7.91,33.096234,15.367341,33.3,16.001257,6.004401,3.839423,False
3,Applied Materials,Equipment,27.32,27.31,-0.01,-0.036603,-0.018303,31.1,12.010253,9.996338,3.296279,False
0,AMD,Fabless,23.00,26.05,3.05,13.260870,6.424090,23.6,20.000000,9.980806,3.144199,False


Top 10 by absolute revenue increase


,company_name,value_chain_group,revenue_2022,revenue_2024,absolute_revenue_change_usd_bn,revenue_growth_2022_2024_pct,two_year_cagr_pct,operating_margin_pct,rd_intensity_pct,capex_intensity_pct,revenue_share_within_dataset_pct,operating_income_review_flag
21,NVIDIA,Fabless,26.12,130.48,104.36,399.540582,123.504045,59.5,25.000000,2.000307,15.748754,False
4,Broadcom,Fabless,33.50,54.34,20.84,62.208955,27.361280,25.4,11.998528,9.992639,6.558762,False
27,SK Hynix,IDM,34.69,51.53,16.84,48.544249,21.878730,28.8,9.994178,35.008733,6.219599,False
34,TSMC,Foundry,74.93,88.89,13.96,18.630722,8.917731,41.3,7.998650,44.999438,10.728899,False
1,ASML,Equipment,23.90,31.81,7.91,33.096234,15.367341,33.3,16.001257,6.004401,3.839423,False
11,HiSilicon (Huawei),Fabless,1.97,9.12,7.15,362.944162,115.161373,30.2,11.951754,9.978070,1.100771,False
0,AMD,Fabless,23.00,26.05,3.05,13.260870,6.424090,23.6,20.000000,9.980806,3.144199,False
39,Yangtze Memory (YMTC),IDM,4.12,6.62,2.50,60.679612,26.759462,31.6,9.969789,35.045317,0.799025,False
7,ChangXin Memory (CXMT),IDM,3.10,5.07,1.97,63.548387,27.886038,30.0,10.059172,34.911243,0.611942,False
28,SMIC,Foundry,7.09,8.20,1.11,15.655853,7.543411,37.3,8.048780,45.000000,0.989729,False


...

,company_name,value_chain_group,revenue_2022,revenue_2024,absolute_revenue_change_usd_bn,revenue_growth_2022_2024_pct,two_year_cagr_pct,operating_margin_pct,rd_intensity_pct,capex_intensity_pct,revenue_share_within_dataset_pct,operating_income_review_flag
21,NVIDIA,Fabless,26.12,130.48,104.36,399.540582,123.504045,59.5,25.000000,2.000307,15.748754,False
36,Texas Instruments,IDM,19.08,14.89,-4.19,-21.960168,-11.659844,42.6,12.021491,12.021491,1.797202,False
2,Analog Devices,IDM,12.20,9.47,-2.73,-22.377049,-11.896112,42.3,12.038015,12.038015,1.143016,False
31,Samsung Foundry,Foundry,18.03,18.09,0.06,0.332779,0.166251,41.8,8.015478,44.997236,2.183438,False
5,Cadence,EDA Software,3.52,4.47,0.95,26.988636,12.689235,41.3,34.899329,0.894855,0.539523,False
34,TSMC,Foundry,74.93,88.89,13.96,18.630722,8.917731,41.3,7.998650,44.999438,10.728899,False
12,Hua Hong,Foundry,2.51,1.94,-0.57,-22.709163,-12.084793,40.3,8.247423,44.845361,0.234155,False
15,KLA,Equipment,9.22,9.96,0.74,8.026030,3.935572,38.1,12.048193,10.040161,1.202158,False
28,SMIC,Foundry,7.09,8.20,1.11,15.655853,7.543411,37.3,8.048780,45.000000,0.989729,False
8,GlobalFoundries,Foundry,8.00,6.54,-1.46,-18.250000,-9.584293,37.0,7.951070,44.954128,0.789369,False


Top 10 by 2024 R&D intensity


,company_name,value_chain_group,revenue_2022,revenue_2024,absolute_revenue_change_usd_bn,revenue_growth_2022_2024_pct,two_year_cagr_pct,operating_margin_pct,rd_intensity_pct,capex_intensity_pct,revenue_share_within_dataset_pct,operating_income_review_flag
6,Cerebras Systems,Fabless,0.22,0.77,0.55,250.000000,87.082869,4.3,50.649351,1.298701,0.092938,False
10,Groq,Fabless,0.02,0.48,0.46,2300.000000,389.897949,8.5,50.000000,0.000000,0.057935,False
30,SambaNova,Fabless,0.05,0.37,0.32,640.000000,172.029410,5.9,48.648649,0.000000,0.044658,False
35,Tenstorrent,Fabless,0.05,0.31,0.26,520.000000,148.997992,11.7,48.387097,0.000000,0.037417,False
9,Graphcore,Fabless,0.05,0.05,0.00,0.000000,0.000000,11.1,40.000000,0.000000,0.006035,False
33,Synopsys,EDA Software,4.82,5.91,1.09,22.614108,10.731255,36.9,35.025381,1.015228,0.713329,False
5,Cadence,EDA Software,3.52,4.47,0.95,26.988636,12.689235,41.3,34.899329,0.894855,0.539523,False
21,NVIDIA,Fabless,26.12,130.48,104.36,399.540582,123.504045,59.5,25.000000,2.000307,15.748754,False
0,AMD,Fabless,23.00,26.05,3.05,13.260870,6.424090,23.6,20.000000,9.980806,3.144199,False
14,Intel,IDM,64.11,54.42,-9.69,-15.114647,-7.866752,8.8,19.992650,25.009188,6.568418,False


Top 10 by 2024 CapEx intensity


,company_name,value_chain_group,revenue_2022,revenue_2024,absolute_revenue_change_usd_bn,revenue_growth_2022_2024_pct,two_year_cagr_pct,operating_margin_pct,rd_intensity_pct,capex_intensity_pct,revenue_share_within_dataset_pct,operating_income_review_flag
38,UMC,Foundry,9.40,7.41,-1.99,-21.170213,-11.213860,35.9,7.962213,45.074224,0.894377,False
28,SMIC,Foundry,7.09,8.20,1.11,15.655853,7.543411,37.3,8.048780,45.000000,0.989729,False
34,TSMC,Foundry,74.93,88.89,13.96,18.630722,8.917731,41.3,7.998650,44.999438,10.728899,False
31,Samsung Foundry,Foundry,18.03,18.09,0.06,0.332779,0.166251,41.8,8.015478,44.997236,2.183438,False
8,GlobalFoundries,Foundry,8.00,6.54,-1.46,-18.250000,-9.584293,37.0,7.951070,44.954128,0.789369,False
12,Hua Hong,Foundry,2.51,1.94,-0.57,-22.709163,-12.084793,40.3,8.247423,44.845361,0.234155,False
39,Yangtze Memory (YMTC),IDM,4.12,6.62,2.50,60.679612,26.759462,31.6,9.969789,35.045317,0.799025,False
20,Micron,IDM,32.09,22.96,-9.13,-28.451231,-15.413495,28.4,10.017422,35.017422,2.771240,False
27,SK Hynix,IDM,34.69,51.53,16.84,48.544249,21.878730,28.8,9.994178,35.008733,6.219599,False
32,Samsung Memory,IDM,75.70,74.79,-0.91,-1.202114,-0.602874,26.8,10.001337,35.004680,9.027049,False


## Revenue concentration

These measures describe concentration only among entities represented in this dataset. HHI uses squared percentage shares.

In [8]:
concentration_records = []
for year, group in df_historical.groupby("year"):
    shares = group["revenue_usd_bn"].div(group["revenue_usd_bn"].sum()).mul(100).sort_values(ascending=False)
    concentration_records.append({"year": year, "largest_company_revenue_share_pct": shares.iloc[0], "top_3_revenue_share_pct": shares.head(3).sum(), "top_5_revenue_share_pct": shares.head(5).sum(), "hhi": shares.pow(2).sum()})
concentration = pd.DataFrame(concentration_records)
display(concentration)

,year,largest_company_revenue_share_pct,top_3_revenue_share_pct,top_5_revenue_share_pct,hhi
0,2010,18.863830,37.800000,48.676596,763.780498
1,2011,18.278623,37.906325,48.795036,744.527664
2,2012,18.997263,38.590768,49.698129,765.545935
3,2013,17.866441,38.150025,50.555907,740.350129
4,2014,17.238788,38.753971,51.296239,741.270232
5,2015,16.920262,38.180534,52.552003,738.128948
6,2016,16.592195,38.616730,51.303924,737.860996
7,2017,15.378181,37.676310,50.837690,725.227230
8,2018,15.949181,37.351953,50.942881,717.608011
9,2019,15.002525,38.074942,49.782850,706.668230


## Investment-growth associations and diagnostic sensitivity

Pearson correlations are descriptive associations only. No causal or statistical-significance claim is made.

In [9]:
correlation_specs = [
    ("2022 R&D intensity vs 2022–2024 CAGR", "rd_intensity_2022_pct", "two_year_cagr_pct"),
    ("2022 CapEx intensity vs 2022–2024 CAGR", "capex_intensity_2022_pct", "two_year_cagr_pct"),
    ("2024 R&D intensity vs 2024 operating margin", "rd_intensity_pct", "operating_margin_pct"),
    ("2024 CapEx intensity vs 2024 operating margin", "capex_intensity_pct", "operating_margin_pct"),
]
correlation_records = []
for label, x_column, y_column in correlation_specs:
    pairs = company_growth[[x_column, y_column]].dropna()
    correlation_records.append({"association": label, "pearson_correlation": pairs[x_column].corr(pairs[y_column]), "sample_size": len(pairs)})
correlations = pd.DataFrame(correlation_records)
display(correlations)
main_2024_ranking = company_growth.nlargest(10, "revenue_2024")
unflagged_2024_ranking = company_growth.loc[~company_growth.operating_income_review_flag].nlargest(10, "revenue_2024")
sensitivity = pd.DataFrame({
    "comparison": ["Top company unchanged", "Top-10 membership unchanged", "Flagged rows excluded"],
    "result": [main_2024_ranking.iloc[0].company_name == unflagged_2024_ranking.iloc[0].company_name, set(main_2024_ranking.company_name) == set(unflagged_2024_ranking.company_name), int(company_growth.operating_income_review_flag.sum())],
})
display(main_2024_ranking[["company_name", "revenue_2024", "operating_income_review_flag"]])
display(unflagged_2024_ranking[["company_name", "revenue_2024", "operating_income_review_flag"]])
display(sensitivity)

,association,pearson_correlation,sample_size
0,2022 R&D intensity vs 2022–2024 CAGR,0.687489,40
1,2022 CapEx intensity vs 2022–2024 CAGR,-0.353822,40
2,2024 R&D intensity vs 2024 operating margin,-0.522154,40
3,2024 CapEx intensity vs 2024 operating margin,0.351341,40


,company_name,revenue_2024,operating_income_review_flag
21,NVIDIA,130.48,False
34,TSMC,88.89,False
32,Samsung Memory,74.79,False
14,Intel,54.42,False
4,Broadcom,54.34,False
27,SK Hynix,51.53,False
24,Qualcomm,39.18,False
1,ASML,31.81,False
3,Applied Materials,27.31,False
0,AMD,26.05,False


,company_name,revenue_2024,operating_income_review_flag
21,NVIDIA,130.48,False
34,TSMC,88.89,False
32,Samsung Memory,74.79,False
14,Intel,54.42,False
4,Broadcom,54.34,False
27,SK Hynix,51.53,False
24,Qualcomm,39.18,False
1,ASML,31.81,False
3,Applied Materials,27.31,False
0,AMD,26.05,False


,comparison,result
0,Top company unchanged,True
1,Top-10 membership unchanged,True
2,Flagged rows excluded,0


## Caution-period appendix

The following 2025–2026 values are included in AI expansion and also reported separately for source review. Other historical analyses retain their stated year windows.

In [10]:
caution_summary = df_caution.groupby("year").agg(total_revenue_usd_bn=("revenue_usd_bn", "sum"), company_count=("company_name", "nunique"), source_validation_caution_row_count=("source_validation_caution", "sum")).reset_index()
caution_value_chain = df_caution.groupby(["year", "value_chain_group"], as_index=False).agg(value_chain_revenue_usd_bn=("revenue_usd_bn", "sum"))
display(caution_summary)
display(caution_value_chain)

,year,total_revenue_usd_bn,company_count,source_validation_caution_row_count
0,2025,1008.04,40,40
1,2026,1150.73,40,40


,year,value_chain_group,value_chain_revenue_usd_bn
0,2025,EDA Software,11.77
1,2025,Equipment,121.27
2,2025,Fabless,371.74
3,2025,Foundry,156.43
4,2025,IDM,346.83
5,2026,EDA Software,13.14
6,2026,Equipment,130.60
7,2026,Fabless,455.02
8,2026,Foundry,164.83
9,2026,IDM,387.14


## EDA Findings Table and Validation

In [11]:
top_2024 = company_growth.nlargest(1, "revenue_2024").iloc[0]
largest_increase = company_growth.nlargest(1, "absolute_revenue_change_usd_bn").iloc[0]
highest_cagr = company_growth.nlargest(1, "two_year_cagr_pct").iloc[0]
top_share_2024 = concentration.loc[concentration.year.eq(2024)].iloc[0]
strongest_association = correlations.loc[correlations.pearson_correlation.abs().idxmax()]
top_group_change_metric = "percentage_difference__average_annual_total_revenue_usd_bn"
top_group_change = period_comparison[top_group_change_metric].idxmax()
eda_findings = pd.DataFrame([
    ["How did dataset revenue change?", "Annual total revenue and YoY growth", f"Highest growth was {int(highest_growth_year.year)} ({highest_growth_year.year_over_year_total_revenue_change_pct:.2f}%); lowest was {int(lowest_growth_year.year)} ({lowest_growth_year.year_over_year_total_revenue_change_pct:.2f}%).", "Totals can reflect dataset coverage as well as company performance.", "Line chart with company-count context"],
    ["Which value-chain group grew most between periods?", "Change in average annual group revenue", f"{top_group_change} had the largest percentage increase in average annual revenue from Smart automobile expansion (2010–2021) to AI expansion (2023–2026).", "Average annual values account for unequal period lengths; 2025–2026 require source validation; no causal attribution.", "Grouped bar chart"],
    ["Which company was largest in 2024?", "2024 revenue within dataset", f"{top_2024.company_name} ranked first with {top_2024.revenue_2024:.2f} USD bn.", "Dataset entities only; not global market share.", "Horizontal bar chart"],
    ["Which company added the most revenue, 2022–2024?", "Absolute revenue change", f"{largest_increase.company_name} increased by {largest_increase.absolute_revenue_change_usd_bn:.2f} USD bn.", "Only companies with positive revenue in both endpoint years.", "Dumbbell chart"],
    ["Which company had the highest two-year CAGR?", "2022–2024 CAGR with revenue context", f"{highest_cagr.company_name} had the highest CAGR ({highest_cagr.two_year_cagr_pct:.2f}%), from {highest_cagr.revenue_2022:.2f} to {highest_cagr.revenue_2024:.2f} USD bn.", "CAGR can amplify small-base effects.", "Scatter plot: starting revenue vs CAGR"],
    ["How concentrated was 2024 dataset revenue?", "Largest, top-3, top-5 shares and HHI", f"Largest={top_share_2024.largest_company_revenue_share_pct:.2f}%, top-3={top_share_2024.top_3_revenue_share_pct:.2f}%, top-5={top_share_2024.top_5_revenue_share_pct:.2f}%, HHI={top_share_2024.hhi:.2f}.", "Concentration only among represented entities.", "Concentration trend lines"],
    ["How are investment and performance associated?", "Descriptive Pearson correlations", f"Largest absolute correlation was {strongest_association.association}: r={strongest_association.pearson_correlation:.3f}, n={int(strongest_association.sample_size)}.", "Association is not causation or statistical significance.", "Scatter plots with sample size"],
    ["Does excluding operating-income review flags change the top ranking?", "Top-10 2024 revenue sensitivity", f"Top company unchanged: {bool(sensitivity.loc[0, 'result'])}; top-10 membership unchanged: {bool(sensitivity.loc[1, 'result'])}.", "Flags indicate review needs and do not prove errors.", "Side-by-side ranking table"],
], columns=["analysis_question", "metric_used", "result_supported_by_data", "important_limitation", "recommended_chart_type"])
display(eda_findings)

company_share_totals = df_historical.groupby("year").revenue_share_within_dataset_pct.sum()
group_share_totals = value_chain_yearly.groupby("year").value_chain_revenue_share_pct.sum()
pd.testing.assert_frame_equal(df, df_snapshot, obj="input df")
eda_validation = {
    "input_row_count_617": len(df) == 617,
    "no_input_rows_changed_or_removed": df.equals(df_snapshot),
    "historical_subset_ends_2024": int(df_historical.year.max()) == 2024,
    "caution_subset_only_2025_2026": set(df_caution.year.unique()) == {2025, 2026},
    "company_shares_sum_approximately_100": company_share_totals.sub(100).abs().le(1e-9).all(),
    "value_chain_shares_sum_approximately_100": group_share_totals.sub(100).abs().le(1e-9).all(),
    "no_infinite_values": not df.select_dtypes(include="number").isin([float("inf"), float("-inf")]).any().any(),
    "correlation_sample_sizes_reported": correlations.sample_size.notna().all() and correlations.sample_size.gt(0).all(),
    "rankings_include_revenue_context": all({"revenue_2022", "revenue_2024"}.issubset(table.columns) for table in rankings.values()),
}
for check, passed in eda_validation.items():
    assert passed, f"EDA validation failed: {check}"
display(pd.Series(eda_validation, name="passed").to_frame())

,analysis_question,metric_used,result_supported_by_data,important_limitation,recommended_chart_type
0,How did dataset revenue change?,Annual total revenue and YoY growth,Highest growth was 2024 (26.62%); lowest was 2...,Totals can reflect dataset coverage as well as...,Line chart with company-count context
1,Which value-chain group grew most between peri...,Change in average annual group revenue,Fabless had the largest percentage increase in...,Average annual values account for unequal peri...,Grouped bar chart
2,Which company was largest in 2024?,2024 revenue within dataset,NVIDIA ranked first with 130.48 USD bn.,Dataset entities only; not global market share.,Horizontal bar chart
3,"Which company added the most revenue, 2022–2024?",Absolute revenue change,NVIDIA increased by 104.36 USD bn.,Only companies with positive revenue in both e...,Dumbbell chart
4,Which company had the highest two-year CAGR?,2022–2024 CAGR with revenue context,"Groq had the highest CAGR (389.90%), from 0.02...",CAGR can amplify small-base effects.,Scatter plot: starting revenue vs CAGR
5,How concentrated was 2024 dataset revenue?,"Largest, top-3, top-5 shares and HHI","Largest=15.75%, top-3=35.50%, top-5=48.63%, HH...",Concentration only among represented entities.,Concentration trend lines
6,How are investment and performance associated?,Descriptive Pearson correlations,Largest absolute correlation was 2022 R&D inte...,Association is not causation or statistical si...,Scatter plots with sample size
7,Does excluding operating-income review flags c...,Top-10 2024 revenue sensitivity,Top company unchanged: True; top-10 membership...,Flags indicate review needs and do not prove e...,Side-by-side ranking table


,passed
input_row_count_617,True
no_input_rows_changed_or_removed,True
historical_subset_ends_2024,True
caution_subset_only_2025_2026,True
company_shares_sum_approximately_100,True
value_chain_shares_sum_approximately_100,True
no_infinite_values,True
correlation_sample_sizes_reported,True
rankings_include_revenue_context,True


## EDA Conclusions and Candidate Visuals

**Confirmed descriptive findings:** Within the 2010–2024 historical subset, total dataset revenue had its highest annual growth in 2024 (26.62%) and its lowest in 2023 (-6.18%). The revised three-period comparison and its calculated leading group are reported in the EDA Findings Table above; AI expansion includes source-review years 2025–2026. NVIDIA ranked first by 2024 revenue (USD 130.48 billion) and recorded the largest absolute 2022–2024 increase (USD 104.36 billion). The 2024 largest-company, top-3, and top-5 dataset revenue shares were 15.75%, 35.50%, and 48.63%, respectively, with an HHI of 672.60. Excluding operating-income review flags did not change the leading company or top-10 membership; there were no flagged 2024 rows in that ranking population.

**Suspected patterns:** Groq had the highest two-year CAGR (389.90%), but its revenue rose from only USD 0.02 billion to USD 0.48 billion, so this is strongly affected by a small starting base. The largest absolute descriptive correlation was between 2022 R&D intensity and 2022–2024 CAGR (r = 0.687, n = 40). This association does not demonstrate causation or statistical significance.

**Data-quality limitations:** Revenue totals, shares, rankings, concentration, and value-chain comparisons cover only entities represented in this dataset and are not global market-share estimates. Coverage changes can affect trends. Operating-income flags indicate review needs rather than proven errors. Results for 2025–2026 are included in the AI expansion period comparison and also shown in the caution appendix; conclusions using that comparison require source validation.

**Recommended final visualizations:** Use a revenue trend line with company-count context, grouped bars for value-chain period comparisons, horizontal company revenue bars, a 2022–2024 revenue dumbbell chart, a starting-revenue-versus-CAGR scatter plot, concentration trend lines, investment-association scatter plots with sample sizes, and a side-by-side sensitivity ranking table. No charts are created here.